# 05_03 – Восстановление отсутствующей классификации ВС

Используем сохраненный официальный снимок ICAO Doc 8643 от 08.10.2026 с проверкой SHA256. Только при отсутствующем icaoaircrafttype и прямом совпадении typecode с обычным Designator, имеющим единственное Description, добавляем производную классификацию. Регистр меняем только для поиска; названия моделей, подстроки и предположительные псевдонимы не используем.

Источник: [ICAO Doc 8643](https://www.icao.int/operational-safety/doc-8643-aircraft-type-designators/search). Историческая применимость справочника, идентичность борта и доступность признака в момент полета не подтверждены. Обратное восстановление typecode по широкой классификации не выполняем. Это справочное обогащение, не допуск поля в модель.

Создаем отдельную копию, сохраняем все строки и исходные поля. Новые поля: aircraft_description_restored, aircraft_restoration_status, aircraft_restoration_source, aircraft_restoration_checked_date.

In [1]:
# Находим проект и применяем однозначное справочное сопоставление
from pathlib import Path
import sys
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'europe/scripts/restore_aircraft_classification.py').exists())
sys.path.insert(0, str(ROOT / 'europe/scripts'))
import restore_aircraft_classification as restoration
summary, details, journal = restoration.main()
display(summary)
print(journal)
# Повторный запуск остановится при существующей выходной копии

,restoration_status,rows,go_arounds
0,both_type_fields_missing,218080,688
1,no_unique_direct_reference_match,41955,105
2,restored_from_unique_designator,21703,105
3,source_classification_present,6049756,17460


{'input': 'europe\\data\\analytical\\04_c40_available.parquet', 'input_sha256': 'f51a710fd9a11fe7003776748c3dc4f5792d81496713d96fb59c8c8f256e717e', 'input_unchanged': True, 'output': 'europe\\data\\analytical\\05_aircraft_classification_enriched.parquet', 'output_sha256': '95e73053a6007df4ade5b84d07ef9ed330e7791d97ff4957793c68b00a13b199', 'rows_before': 6331494, 'rows_after': 6331494, 'go_arounds_before': 18358, 'go_arounds_after': 18358, 'rows_deleted': 0, 'source_values_changed': 0, 'classifications_restored': 21703, 'positive_classifications_restored': 105, 'classification_missing_after': 260035, 'positive_classification_missing_after': 793, 'added_fields': ['aircraft_description_restored', 'aircraft_restoration_status', 'aircraft_restoration_source', 'aircraft_restoration_checked_date'], 'original_fields_exact_comparison_passed': True, 'rule': 'Missing icaoaircrafttype only; uppercase typecode exactly matches an ordinary ICAO Designator with one Description', 'reference_url': 'http

In [2]:
# Показываем восстановленные коды и остаток частично заполненной группы
display(details[details.restoration_status.eq('restored_from_unique_designator')].sort_values('go_arounds',ascending=False))
display(details[details.restoration_status.eq('no_unique_direct_reference_match')].sort_values('go_arounds',ascending=False))

,typecode,restoration_status,description_restored,rows,go_arounds
178,E290,restored_from_unique_designator,L2J,14570,82
179,E295,restored_from_unique_designator,L2J,7133,23


,typecode,restoration_status,description_restored,rows,go_arounds
240,GL6T,no_unique_direct_reference_match,NaN,6666,20
24,A320-251N,no_unique_direct_reference_match,NaN,4318,15
26,A321-231,no_unique_direct_reference_match,NaN,1324,9
228,G550,no_unique_direct_reference_match,NaN,3896,9
150,CL65,no_unique_direct_reference_match,NaN,3466,8
198,F2LX,no_unique_direct_reference_match,NaN,3092,8
151,CL85,no_unique_direct_reference_match,NaN,1522,5
232,G650,no_unique_direct_reference_match,NaN,2732,5
42,A380,no_unique_direct_reference_match,NaN,338,4
197,F2EX,no_unique_direct_reference_match,NaN,3324,4


## Результат

Восстановлено 21 703 классификации, включая 105 положительных наблюдений. Все строки и исходные поля сохранены. Из 911 положительных наблюдений с пропусками у 806 остается пропуск хотя бы одного поля. [Результаты, ограничения и актуальная копия](../../reports/aircraft_types/05_03_review.md).